# nano-laser vs laser_encoders on every LASER3 model

Runs `scripts/check_laser3.py`: each of the 147 LASER3 checkpoints is downloaded, encoded by both packages, and
deleted. About 1 to 2 hours on a T4 (Runtime → Change runtime type → T4 GPU).

Results go to a JSONL file, one line per language. Save it on Google Drive (cell 3) and an interrupted run resumes
where it stopped when the notebook is run again.

## 1. nano-laser

In [ ]:
!nvidia-smi -L
!git clone -q -b chore/check-laser3 https://github.com/BurkimbIA/nano-laser.git /content/nano-laser
!pip install -q -e /content/nano-laser

## 2. laser_encoders, in its own Python 3.10 environment

fairseq 0.12.2 does not install on Colab's Python, so the reference runs in a separate env built with uv.
fairseq compiles from source: this cell takes about 5 minutes.

In [ ]:
!pip install -q uv
!uv venv -q --python 3.10 /content/ref
!uv pip install -q --python /content/ref/bin/python "pip<24" "setuptools<70" wheel cython "numpy<2" torch==2.1.2 requests tqdm
!/content/ref/bin/python -m pip install -q fairseq==0.12.2 laser_encoders==0.0.2
!/content/ref/bin/python -c "import laser_encoders, fairseq; print('laser_encoders ok, fairseq', fairseq.__version__)"

## 3. Where to keep the results

In [ ]:
from google.colab import drive

drive.mount("/content/drive")
RESULTS = "/content/drive/MyDrive/nano-laser/laser3_parity.jsonl"
!mkdir -p /content/drive/MyDrive/nano-laser

## 4. Run

Add `--langs mos_Latn dyu_Latn` to try a few languages first.

In [ ]:
!cd /content/nano-laser && python scripts/check_laser3.py --ref-python /content/ref/bin/python --results "$RESULTS" --model-dir /content/laser-models 2>&1 | grep -v "already downloaded"

## 5. Summary

In [ ]:
import pandas as pd

results = pd.read_json(RESULTS, lines=True)
print(f"{results.ok.sum()} / {len(results)} languages ok")
cfg = ["arch", "encoder_layers", "encoder_embed_dim", "encoder_attention_heads", "encoder_normalize_before",
       "activation_fn", "sentemb_criterion", "prepend_bos", "left_pad_source"]
display(results.groupby(cfg, dropna=False).size().rename("languages").reset_index())
display(results[["min_cosine_bs1", "min_cosine_bs8", "max_rel_diff_bs1", "max_rel_diff_bs8"]].describe())
display(results[~results.ok])